# Case study: morbidity rates in Papua

Estimate the 2023 morbidity rate of every district and city on Papua Island precisely
enough to publish, working through one complete analysis from the published survey
figures to a table you could hand over.

The analysis follows Prayoga, Pusponegoro, Sukim and Budiarti (2024), *Small area
estimation for morbidity rate prediction*, Commun. Math. Biol. Neurosci. 2024:62
([doi:10.28919/cmbn/8845](https://doi.org/10.28919/cmbn/8845)), which fits a
hierarchical Bayes (HB) Beta-logistic model to the same data. It is organised by the six
phases of CRISP-DM:

| Phase | hbsaemp | Other libraries |
| --- | --- | --- |
| 1 · Business understanding | — | — |
| 2 · Data understanding | `load_dataset`, `REAL_DATASETS` | pandas, NumPy, matplotlib |
| 3 · Data preparation | — | pandas, NumPy, SciPy, matplotlib |
| 4 · Modeling | `get_family_spec`, `ModelConfig`, `hbm_beta`, `check_data`, `check_prior`, `fit` | — |
| 5 · Evaluation | `check_convergence`, `update_model`, `compare_models`, `estimate_areas` | NumPy, pandas, matplotlib |
| 6 · Deployment | the `estimate_areas` result; `update_model` for the next round | pandas, matplotlib |

You need the package installed with its modelling extra, as shown on the
{doc}`home page <../../index>`. Two models are sampled, so the notebook takes a few
minutes to run.

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

import hbsaemp as hb

## 1 · Business understanding

**The problem.** The morbidity rate is the percentage of the population with a health
complaint in the past month that disrupted their daily activities. It is one of the
indicators behind non-communicable disease control, which Law No. 17 of 2023 makes a
regional priority, so each district needs a rate of its own.

The rate is estimated directly from the National Socio-Economic Survey (Susenas). In a
district with few sampled households the direct estimate loses precision, and Prayoga et
al. single out Papua Island as the region where this happens: in 2023, 7 of its 42
districts and cities have a relative standard error (RSE) above 25%.

**The objective.** A 2023 morbidity rate for all 42 districts and cities, each precise
enough to publish.

**The success criterion.** Statistics Indonesia (BPS) grades an estimate by its RSE:

| RSE | Reading |
| --- | --- |
| ≤ 25% | precise; publish as is |
| 25% – 50% | usable, with caution in interpretation |
| > 50% | imprecise |

The objective is met when every district lands in the first class.

**The approach.** Small area estimation borrows strength from auxiliary variables
available for every district — here, village infrastructure counted by the 2021 Village
Potential Data Collection (Podes) — through a hierarchical Bayes model. Because the rate
is a proportion, the model is a Beta likelihood with a logit link: the HB Beta-logistic
model of Prayoga et al.

## 2 · Data understanding

*hbsaemp in this phase:* `load_dataset`, `REAL_DATASETS`.

### Collect the data

Both sources ship with the package, exactly as published. `REAL_DATASETS` lists them;
their full data dictionary — sources, units, and how each Podes variable was computed from
the village records — is in {doc}`../reference/datasets`.

In [ ]:
print(hb.REAL_DATASETS)

susenas = hb.load_dataset("susenas2023_papua")
podes = hb.load_dataset("podes2021_papua")
print(susenas.shape, podes.shape)

`susenas2023_papua` holds the direct estimates: one morbidity rate per district, in
percent, with its standard error and RSE.

In [ ]:
susenas.head()

`podes2021_papua` holds ten auxiliary variables, `X1` to `X10`, for the same districts:
three on sanitation and water (percentages of villages), three on public schools, and
four on health facilities and workers (averages per village). The first columns identify
the district and its province before and after the 2022 provincial split.

In [ ]:
podes.head()

### Describe the data

Start with the variable of interest. One district has no rate:

In [ ]:
susenas[susenas["morbidity_rate"].isna()]

In [ ]:
susenas["morbidity_rate"].describe().round(2)

Classify the direct estimates by the success criterion. The helper is reused in the
evaluation phase:

In [ ]:
def precision_class(rse):
    '''BPS precision class of a relative standard error given in percent.'''
    return pd.cut(rse, bins=[0, 25, 50, np.inf],
                  labels=["RSE ≤ 25%", "25% < RSE ≤ 50%", "RSE > 50%"])


precision_class(susenas["rse"]).value_counts().sort_index()

35 districts are precise, 6 need caution and 1 is imprecise — the same counts as the
direct-estimation column of Prayoga et al. (2024), Table 4. The imprecise one is
Deiyai, the district without a rate.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(susenas["morbidity_rate"].dropna(), bins=12)
axes[0].set_xlabel("Morbidity rate (%)")
axes[0].set_ylabel("Districts")
axes[1].hist(susenas["rse"], bins=12)
axes[1].axvline(25, color="tab:red", linestyle="--", label="RSE = 25%")
axes[1].set_xlabel("RSE of the direct estimate (%)")
axes[1].legend()
fig.tight_layout()

Then the auxiliary variables:

In [ ]:
aux = [f"X{i}" for i in range(1, 11)]
podes[aux].describe().T.round(3)

`X3` barely varies — in every district at least 98% of villages bathe and wash with water
from a decent source — so it can carry little information about differences between
districts.

### Verify data quality

Before combining the two frames, check that their district codes agree:

In [ ]:
keys = susenas[["idkab", "nama_kab"]].merge(
    podes[["idkab", "nama_kab"]], on="idkab", how="outer",
    suffixes=("_susenas", "_podes"), indicator=True,
)
keys[keys["_merge"] != "both"]

Two problems, both to be fixed in the next phase:

1. **Kota Jayapura** is published as 9437 in Susenas but has the official code 9471 in
   Podes, so a merge on the code would silently drop it.
2. **Deiyai** has no morbidity rate and no standard error.

A third point is documented rather than fixed: four village records carry the value 99 in
a health-worker item, which feeds `X9` for Biak Numfor and `X10` for three other
districts. Whether 99 is a real count could not be verified, so the aggregates are used as
recorded (see the note in {doc}`../reference/datasets`).

## 3 · Data preparation

*hbsaemp in this phase:* none — this is pandas, NumPy and SciPy work, producing the one
frame the model will read.

### Clean

Recode Kota Jayapura to its official code. For Deiyai, use the value Prayoga et al.
(2024) model: their Table 1 gives a minimum rate of 0.2958%, which is Deiyai's, since
every other district is above 1%. Its standard error then follows from the RSE that
*is* published, 60.63%.

In [ ]:
susenas_clean = susenas.copy()
susenas_clean["idkab"] = susenas_clean["idkab"].replace({9437: 9471})

deiyai = susenas_clean["nama_kab"] == "Deiyai"
susenas_clean.loc[deiyai, "morbidity_rate"] = 0.2958
susenas_clean.loc[deiyai, "se"] = 0.2958 * susenas_clean.loc[deiyai, "rse"] / 100
susenas_clean[deiyai]

With Deiyai in place, the descriptive statistics reproduce those of the paper, up to the
two decimals the published rates are rounded to:

In [ ]:
summary = susenas_clean["morbidity_rate"].agg(["min", "mean", "max", "var"])
summary["range"] = summary["max"] - summary["min"]
pd.DataFrame({
    "this data": summary.round(4),
    "Prayoga et al. (2024), Table 1": [0.2958, 5.9079, 13.7658, 8.6201, 13.4699],
})

Treating Deiyai as a district with no direct estimate at all would be the alternative:
leave it out of the fit and predict it afterwards with `estimate_areas(new_data=...)`.
This tutorial keeps the 42 districts of the paper instead.

### Integrate

In [ ]:
data = susenas_clean.merge(podes.drop(columns="nama_kab"), on="idkab", validate="one_to_one")
data.shape

### Construct

The model needs the rate as a proportion, $\hat\theta_i$, and the precision of each
direct estimate, $\phi_i$. The Beta sampling model of Prayoga et al. (their Eq. 7) is

$$
\hat\theta_i \mid \theta_i \sim \text{Beta}\big(\theta_i \phi_i,\; (1-\theta_i)\phi_i\big),
\qquad
\operatorname{Var}(\hat\theta_i) = \frac{\theta_i (1-\theta_i)}{\phi_i + 1},
$$

where the paper takes $\phi_i = n_i / \text{deff}_i - 1$ from the sample size and design
effect. The published tables give neither, only the standard error. Solving the variance
for $\phi_i$, with the direct estimate in place of $\theta_i$, gives

$$
\phi_i = \frac{\hat\theta_i (1-\hat\theta_i)}{\mathrm{SE}_i^2} - 1 .
$$

In [ ]:
data["theta"] = data["morbidity_rate"] / 100
data["phi"] = data["theta"] * (1 - data["theta"]) / (data["se"] / 100) ** 2 - 1

assert data["theta"].between(0, 1, inclusive="neither").all()
assert (data["phi"] > 0).all()
data[["theta", "phi"]].describe().round(4)

### Select the auxiliary variables

Look at each auxiliary variable against the rate first (compare Figure 1 of the paper):

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 5.5), sharey=True)
for ax, x in zip(axes.flat, aux):
    ax.scatter(data[x], data["morbidity_rate"], s=12)
    ax.set_xlabel(x)
for ax in axes[:, 0]:
    ax.set_ylabel("Morbidity rate (%)")
fig.tight_layout()

Then test each correlation, next to the values the paper reports:

In [ ]:
prayoga_r = {"X1": 0.2328, "X2": 0.1902, "X3": -0.2649, "X4": 0.4022, "X5": 0.4198,
             "X6": 0.3339, "X7": 0.4612, "X8": 0.4026, "X9": 0.3107, "X10": 0.3614}

rows = []
for x in aux:
    r, p = pearsonr(data[x], data["morbidity_rate"])
    rows.append({"variable": x, "r": r, "p_value": p, "significant": p < 0.05,
                 "r, Prayoga et al. (2024) Table 2": prayoga_r[x]})
correlation = pd.DataFrame(rows).set_index("variable").round(4)
correlation

The same seven variables, `X4` to `X10`, are significant at the 5% level as in the paper,
and six coefficients agree with it to three or four decimals. Four do not: `X2`, `X7`,
`X8` and `X9`. The paper does not report how it aggregated the village records, so the
cause of the difference cannot be established here; the definitions used in this data are
in {doc}`../reference/datasets`.

Choose among the seven by stepwise selection on AIC, as the paper does. The two
functions below reproduce R's `extractAIC()` for a linear model and `step()` in both
directions:

In [ ]:
def aic(columns):
    '''AIC of an OLS fit of the morbidity rate, as R's extractAIC() computes it.'''
    X = np.column_stack([np.ones(len(data)), *(data[c] for c in columns)])
    beta, *_ = np.linalg.lstsq(X, data["morbidity_rate"], rcond=None)
    rss = np.sum((data["morbidity_rate"] - X @ beta) ** 2)
    return len(data) * np.log(rss / len(data)) + 2 * X.shape[1]


def stepwise(start, scope):
    '''Add or drop one variable at a time while AIC improves, like R's step().'''
    current = list(start)
    while True:
        moves = [[c for c in current if c != drop] for drop in current]
        moves += [current + [add] for add in scope if add not in current]
        best = min(moves, key=aic)
        if aic(best) >= aic(current):
            return current
        current = best


significant = correlation.index[correlation["significant"]].tolist()
selected = stepwise(start=significant, scope=significant)
from_empty = stepwise(start=[], scope=significant)

pd.Series({
    "stepwise from the full model: " + ", ".join(selected): aic(selected),
    "stepwise from the empty model: " + ", ".join(from_empty): aic(from_empty),
    "Prayoga et al. (2024): X7, X8, X9": aic(["X7", "X8", "X9"]),
}, name="AIC").round(2)

Starting from the full model, as R's `step()` is usually applied, keeps `X5`, `X8` and
`X9`. Starting from the empty model keeps `X5` alone at a slightly lower AIC — a reminder
that stepwise selection depends on where it starts. The paper's choice, `X7`, `X8` and
`X9`, is not what this data selects.

This tutorial takes the selection from the full model as its main model, and carries the
paper's variables forward as a second model, so the evaluation phase can compare the two.

## 4 · Modeling

*hbsaemp in this phase:* `get_family_spec`, `ModelConfig`, `hbm_beta`, `check_data`,
`check_prior`, `fit`.

### Select the modeling technique

The Beta family is the one for proportions. Its registry entry confirms the logit link
of the paper's Eq. 8 as the default, and that its precision — `kappa`, as Bambi names
$\phi$ — can be pinned to known values:

In [ ]:
spec = hb.get_family_spec("beta")
print("default link :", spec.default_link)
print("links        :", sorted(spec.supported_links))
print("pinnable     :", spec.pinnable_params)

### Build the models

`hbm_beta` is the beginner interface: a response, a list of auxiliary variables and the
Beta family's own arguments, with the formula written for you (see
{doc}`../explanation/three-tier-api`). `area_var` adds the district random effect $v_i$
of the paper's model,

$$
\operatorname{logit}(\theta_i) = \mathbf{x}_i^\top \boldsymbol\beta + v_i,
\qquad v_i \sim N(0, \sigma_v^2),
$$

and `fixed_params` pins each district's $\phi_i$ to the column built above instead of
estimating it. Both models share one sampler configuration: the default draws, tuning
steps and acceptance target, four chains run in parallel, and a fixed seed so the run can
be repeated:

In [ ]:
config = hb.ModelConfig(chains=4, cores=4, random_seed=2023, progressbar=False)

model = hb.hbm_beta("theta", selected, data, area_var="nama_kab",
                    fixed_params={"kappa": "phi"}, config=config)
model_paper = hb.hbm_beta("theta", ["X7", "X8", "X9"], data, area_var="nama_kab",
                          fixed_params={"kappa": "phi"}, config=config)
print(model.formula)
print(model_paper.formula)

Check the frame the model will actually use before paying for any sampling. The pinned
precision becomes one extra column, an offset on the log scale:

In [ ]:
checked = model.check_data()
print(checked.shape)
print([c for c in checked.columns if c not in data.columns])

### Check the priors

The coefficients and $\sigma_v$ keep hbsaemp's default priors (Bambi's weakly informative
ones). The paper's prior settings — an "initial value" of $N(0, 1)$ for $\boldsymbol\beta$
and a gamma prior on $\sigma_v^{-2}$ whose hyperparameters are not reported — cannot be
reproduced exactly, so the check that matters is that the defaults allow the rates this
data can plausibly take. A prior predictive check samples from the priors alone, without
fitting:

In [ ]:
prior = hb.check_prior(model)
print(prior.summary())

The draws from the priors (blue) spread over the whole unit interval, while the observed
rates (black) sit below 0.15. The default priors are vague: they allow the data without
favouring it, and leave the estimates to the likelihood.

### Fit

Sampling is the expensive step:

In [ ]:
model.fit()
model_paper.fit()
print(model.is_fitted, model_paper.is_fitted)

## 5 · Evaluation

*hbsaemp in this phase:* `check_convergence`, `update_model`, `compare_models`,
`estimate_areas`.

### Convergence

Nothing downstream is trustworthy until the chains have converged. `check_convergence`
reports R-hat, effective sample sizes and the NUTS sampler checks, and warns on any
failure:

In [ ]:
for name, m in [("main model", model), ("paper's variables", model_paper)]:
    conv = hb.check_convergence(m, plot_types=[])
    print(name.upper())
    print(conv.summary(), end="\n\n")

Read the status line of each summary. Divergent transitions mean the sampler failed to
follow the posterior somewhere, so the draws near there are biased; the remedy is a
smaller step size, through a higher `target_accept`. An R-hat above 1.01 means the chains
do not yet agree; more tuning and more draws is the remedy for that.

`update_model` refits in place, changing only the settings you pass. Apply both remedies
to both models, so they stay comparable:

In [ ]:
for m in (model, model_paper):
    hb.update_model(m, target_accept=0.95, tune=2000, draws=2000)

Check again. This time keep the trace and autocorrelation plots, the diagnostics of the
paper's Figure 2:

In [ ]:
print(hb.check_convergence(model_paper, plot_types=[]).summary(), end="\n\n")
conv = hb.check_convergence(model, plot_types=["trace", "acf"])
print(conv.summary())

Chains that have converged overlap without drifting in the trace plot, and their
autocorrelation dies out within a few lags. `kappa_Intercept` is the intercept of the
pinned precision, held at zero by a tight prior — that is how the pin is enforced, so it
is expected to sit there.

### Coefficients

Summarise each coefficient by its posterior mean, standard deviation and 95% credible
interval, as the paper's Table 3 does:

In [ ]:
az.summary(model.result.idata, var_names=["Intercept", *selected, "1|nama_kab_sigma"],
           kind="stats", ci_prob=0.95, ci_kind="eti")

`eti95_lb` and `eti95_ub` bound the central 95% credible interval, and
`1|nama_kab_sigma` is $\sigma_v$, the standard deviation of the district effects. A
coefficient whose interval lies entirely on one side of zero has a clear direction; one
whose interval spans zero is not distinguishable from no effect at this sample size.

For the model with the paper's variables, the paper's own estimates sit alongside:

In [ ]:
paper_table3 = pd.DataFrame(
    {"mean": [-3.2000, 0.8572, 0.5247, -0.4953], "sd": [0.0037, 0.0061, 0.0024, 0.0022],
     "eti95_lb": [-3.2070, 0.8450, 0.5199, -0.4995],
     "eti95_ub": [-3.1928, 0.8693, 0.5293, -0.4911]},
    index=["Intercept", "X7", "X8", "X9"],
)
this_run = az.summary(model_paper.result.idata, var_names=["Intercept", "X7", "X8", "X9"],
                      kind="stats", ci_prob=0.95, ci_kind="eti")
pd.concat({"this run": this_run, "Prayoga et al. (2024), Table 3": paper_table3}, axis=1)

The posterior standard deviations here are far larger than the paper's. The two analyses
differ in several ways, and the paper alone does not show which of them accounts for it:

- the values of `X7`, `X8` and `X9` (their correlations differ, as shown in data
  preparation);
- the precision $\phi_i$, derived here from the standard error and in the paper from the
  sample size and design effect;
- the sampler, NUTS (PyMC) here and Metropolis-Hastings within Gibbs over 100,000
  iterations in the paper;
- the priors, as described under *Check the priors*.

### Compare the two models

`compare_models` ranks the models by leave-one-out cross-validation (PSIS-LOO), and on
request adds a Bayes factor for every coefficient and a prior sensitivity check:

In [ ]:
cmp = hb.compare_models([model, model_paper], metrics=["loo", "bf"],
                        run_prior_sensitivity=True)
plt.close(cmp.params_plot)  # its marginal posteriors repeat the coefficient table
print(cmp.summary())
cmp.comparison_table

`model_0` is the main model and `model_1` the one with the paper's variables. The first
plot is a posterior predictive check of `model_0`: the observed rates (black) should run
inside the rates simulated from the fitted model (blue). The second places the two models
on the ELPD scale, higher being better.

Read the ELPD difference against its standard error: a difference smaller than about
twice its standard error means this data cannot tell the models apart. Pareto *k* values
above the threshold flag observations whose leave-one-out estimate is unreliable, which
is common when every district has one observation and its own random effect; when most
observations are flagged, treat the ranking as indicative only
(see {doc}`../how-to/04-compare-models`).

The Bayes factor tests each coefficient against zero within its model. It is only as
meaningful as the coefficient's prior, and a vague prior, such as the defaults here,
tilts it toward zero:

In [ ]:
cmp.bayes_factor["model_0"]

The prior sensitivity check power-scales the prior and the likelihood and reports how
much each posterior moves; the district effects are left out of the table below for
brevity. Expect `kappa_Intercept` to be flagged as *strong prior / weak likelihood*: its
tight prior is the pin itself.

In [ ]:
sensitivity = cmp.prior_sensitivity["model_0"]
sensitivity[~sensitivity.index.str.startswith("1|nama_kab[")]

### Estimate the districts

`estimate_areas` turns the posterior into one estimate per district, with its
uncertainty:

In [ ]:
est = hb.estimate_areas(model)
print(est.summary())

results = data[["nama_prov", "nama_kab", "morbidity_rate", "rse"]].merge(
    est.result_table, on="nama_kab")
results["hb_rate"] = results["mean"] * 100
results.head()

### Assess the results against the success criterion

Compare the precision classes before and after modelling, next to the paper's result:

In [ ]:
pd.DataFrame({
    "direct": precision_class(results["rse"]).value_counts().sort_index(),
    "HB, this run": precision_class(results["rse_pct"]).value_counts().sort_index(),
    "HB, Prayoga et al. (2024) Table 4": [42, 0, 0],
})

In [ ]:
reached = (results["rse_pct"] <= 25).sum()
print(f"{reached} of {len(results)} districts reach RSE ≤ 25%.")
results.loc[results["rse_pct"] > 25,
            ["nama_kab", "morbidity_rate", "rse", "hb_rate", "rse_pct"]] \
    .sort_values("rse_pct", ascending=False)

Set the two estimators side by side, as the paper's Figure 3 does; the lines mark the
provincial rates it reports, 5.73% for Papua and 6.70% for Papua Barat:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([results["morbidity_rate"], results["hb_rate"]])
ax.set_xticks([1, 2], ["Direct estimate", "HB Beta-logistic"])
ax.axhline(5.73, color="tab:green", linewidth=1, label="Papua, 5.73%")
ax.axhline(6.70, color="tab:blue", linewidth=1, label="Papua Barat, 6.70%")
ax.set_ylabel("Morbidity rate (%)")
ax.legend()
fig.tight_layout()

Unlike the paper's Figure 3, the two boxes are nearly the same: the model estimates are
not visibly more homogeneous than the direct ones. The reason is in the data. The
precisions $\phi_i$ implied by the published standard errors are high (see *Construct*),
so the model treats most direct estimates as already informative and moves them little.

A shrinkage plot shows where it does move them. A district on the diagonal kept its
direct estimate; the further a point sits from it, the more the model pulled that
district toward what its auxiliary variables predict:

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
points = ax.scatter(results["morbidity_rate"], results["hb_rate"], c=results["rse"],
                    cmap="viridis")
ax.plot([0, 15], [0, 15], color="grey", linewidth=1)
ax.set_xlabel("Direct estimate (%)")
ax.set_ylabel("HB estimate (%)")
fig.colorbar(points, label="RSE of the direct estimate (%)")
fig.tight_layout()

The points pulled furthest are the light ones, the districts whose direct estimate is
least precise — Deiyai above all. That is where borrowing strength pays: the gain in
precision comes from the districts that needed it.

Compare the count of precise districts with the objective set in phase 1. A district
still above 25% is not dropped; it is carried into deployment with its precision class.

## 6 · Deployment

*hbsaemp in this phase:* the `estimate_areas` result; `update_model` when the next survey
round arrives.

### The deliverable

The estimates come from the main model, the one this data selected. The comparison in
phase 5 is where a clear reason to prefer the other would have shown: an ELPD difference
well beyond twice its standard error.

One row per district: the direct estimate and the model estimate, each with its
precision, and the BPS class that tells a reader how far to trust it:

In [ ]:
table = pd.DataFrame({
    "province": results["nama_prov"],
    "district": results["nama_kab"],
    "direct (%)": results["morbidity_rate"],
    "direct RSE (%)": results["rse"],
    "HB (%)": results["hb_rate"],
    "95% CI lower (%)": results["ci_lower"] * 100,
    "95% CI upper (%)": results["ci_upper"] * 100,
    "HB RSE (%)": results["rse_pct"],
    "precision": precision_class(results["rse_pct"]),
}).sort_values(["province", "district"]).round(2).reset_index(drop=True)
table

Districts outside the first class stay in the table: publish them with their precision
class rather than presenting them as equally reliable.

The paper maps these estimates (its Figure 4). No boundary files ship with the package, so
rank the districts instead, each with its 95% credible interval:

In [ ]:
ranked = table.sort_values("HB (%)")
fig, ax = plt.subplots(figsize=(7, 10))
ax.errorbar(ranked["HB (%)"], ranked["district"],
            xerr=[ranked["HB (%)"] - ranked["95% CI lower (%)"],
                  ranked["95% CI upper (%)"] - ranked["HB (%)"]],
            fmt="o", markersize=4, capsize=2)
ax.set_xlabel("HB estimate of the morbidity rate, 2023 (%)")
fig.tight_layout()

Kepulauan Yapen remains the district with the highest rate, as it is in the paper. The
width of each interval is the precision a reader should attach to its point.

### Hand it over

Write the table out for whoever publishes it:

```python
table.to_csv("morbidity_papua_2023_hb.csv", index=False)
```

### Keep it current

When the next Susenas round is published, prepare its direct estimates exactly as in
phase 3 and refit the same specification against them, then repeat phase 5 before using a
single number:

```python
hb.update_model(model, new_data=next_round)
```

`update_model` keeps the formula, the family and the pinned precision; the new frame needs
the same columns, including a freshly computed `phi`.

### Next

- {doc}`../how-to/05-estimate-areas` — the estimation step on its own, including districts
  without a direct estimate.
- {doc}`../reference/datasets` — the data dictionary of both datasets.
- {doc}`../explanation/hb-sae` — why borrowing strength reduces the error.